## Phase 1:
shared goal-conditioned observation and SAC+HER agent across Reach/Push/Pick-and-place
random/hand-crafted-curriculum baselines
an adaptive non-LLM curriculum baseline 

In [1]:
import time
import copy
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import gymnasium as gym
from gymnasium import spaces
import panda_gym 

from stable_baselines3 import SAC
from stable_baselines3.her import HerReplayBuffer
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple



_trapz = getattr(np, "trapezoid", None) or np.trapz

print("gymnasium:", gym.__version__)


gymnasium: 1.3.0


## 
1° Task Ladder

In [2]:
TASK_NAMES = ["reach", "push", "slide", "pickandplace", "stack"]
TASK_ENV_IDS = {
    "reach": "PandaReach-v3",
    "push": "PandaPush-v3",
    "slide": "PandaSlide-v3",
    "pickandplace": "PandaPickAndPlace-v3",
    "stack": "PandaStack-v3",
}

MAX_ROBOT_OBS_DIM = 7   
MAX_GOAL_DIM = 6        # Stack's achieved/desired goal is 6D (object1 pos + object2 pos); others padded with zeros
N_TASKS = len(TASK_NAMES)

SEEDS=[10,11,12,13,14]
@dataclass
class TaskSpec:
    name: str
    task_id: int
    env_id: str
    block_gripper: bool
    goal_dim: int               
    threshold: float              # native success distance threshold
    goal_low: np.ndarray              
    goal_high: np.ndarray
    stack_z: Optional[Tuple[float, float]] = None  # (z1, z2) fixed z-offsets, stack only


def build_task_specs() -> Dict[str, TaskSpec]:
    specs = {}

    # Reach: goal_range=0.3 -> low=[-0.15,-0.15,0], high=[0.15,0.15,0.3] (absolute; no separate base point)
    specs["reach"] = TaskSpec(
        "reach", 0, TASK_ENV_IDS["reach"], block_gripper=True, goal_dim=3, threshold=0.05,
        goal_low=np.array([-0.15, -0.15, 0.0]), goal_high=np.array([0.15, 0.15, 0.3]),
    )

    # Push: base=[0,0,object_size/2=0.02], noise in [-0.15,-0.15,0]..[0.15,0.15,0]
    base = np.array([0.0, 0.0, 0.02])
    specs["push"] = TaskSpec(
        "push", 1, TASK_ENV_IDS["push"], block_gripper=True, goal_dim=3, threshold=0.05,
        goal_low=base + np.array([-0.15, -0.15, 0.0]), goal_high=base + np.array([0.15, 0.15, 0.0]),
    )

    # Slide: base=[0,0,object_size/2=0.03], noise in [-0.15+0.4,-0.15,0]..[0.15+0.4,0.15,0]
    base = np.array([0.0, 0.0, 0.03])
    specs["slide"] = TaskSpec(
        "slide", 2, TASK_ENV_IDS["slide"], block_gripper=True, goal_dim=3, threshold=0.05,
        goal_low=base + np.array([-0.15 + 0.4, -0.15, 0.0]), goal_high=base + np.array([0.15 + 0.4, 0.15, 0.0]),
    )

    # Pick-and-place: base=[0,0,object_size/2=0.02], noise in [-0.15,-0.15,0]..[0.15,0.15,0.2]
    base = np.array([0.0, 0.0, 0.02])
    specs["pickandplace"] = TaskSpec(
        "pickandplace", 3, TASK_ENV_IDS["pickandplace"], block_gripper=False, goal_dim=3, threshold=0.05,
        goal_low=base + np.array([-0.15, -0.15, 0.0]), goal_high=base + np.array([0.15, 0.15, 0.2]),
    )

    # Stack: TRUE degrees of freedom are a shared 2D (dx, dy) shift applied to BOTH cubes
   
    specs["stack"] = TaskSpec(
        "stack", 4, TASK_ENV_IDS["stack"], block_gripper=False, goal_dim=2, threshold=0.1,
        goal_low=np.array([-0.15, -0.15]), goal_high=np.array([0.15, 0.15]),
        stack_z=(0.02, 0.06),
    )
    return specs


TASK_SPECS = build_task_specs()


def expand_stack_goal(xy_shift: np.ndarray, spec: TaskSpec) -> np.ndarray:
    """(dx, dy) -> 6D [obj1_x,obj1_y,obj1_z, obj2_x,obj2_y,obj2_z]."""
    z1, z2 = spec.stack_z
    return np.array([xy_shift[0], xy_shift[1], z1, xy_shift[0], xy_shift[1], z2], dtype=np.float32)


def native_goal_dim(name: str) -> int:
    """Dimensionality of the goal as the underlying panda-gym env actually expects it
    (3 for everything except Stack, which is 6)."""
    return 6 if name == "stack" else TASK_SPECS[name].goal_dim


for name, spec in TASK_SPECS.items():
    print(f"{name:14s} env={spec.env_id:22s} block_gripper={spec.block_gripper!s:5s} "
          f"goal_dim={spec.goal_dim} threshold={spec.threshold} "
          f"box=[{spec.goal_low.round(3)}, {spec.goal_high.round(3)}]")


reach          env=PandaReach-v3          block_gripper=True  goal_dim=3 threshold=0.05 box=[[-0.15 -0.15  0.  ], [0.15 0.15 0.3 ]]
push           env=PandaPush-v3           block_gripper=True  goal_dim=3 threshold=0.05 box=[[-0.15 -0.15  0.02], [0.15 0.15 0.02]]
slide          env=PandaSlide-v3          block_gripper=True  goal_dim=3 threshold=0.05 box=[[ 0.25 -0.15  0.03], [0.55 0.15 0.03]]
pickandplace   env=PandaPickAndPlace-v3   block_gripper=False goal_dim=3 threshold=0.05 box=[[-0.15 -0.15  0.02], [0.15 0.15 0.22]]
stack          env=PandaStack-v3          block_gripper=False goal_dim=2 threshold=0.1 box=[[-0.15 -0.15], [0.15 0.15]]


## 2° Held-out evaluation sub-regions (Section 6)

5 small, scattered evaluation zones per task, spanning **near / near / medium / medium /
far** difficulty (distance from the task's own goal-range centre), fixed **once** via a
hardcoded seed so every condition, every seed, every checkpoint is scored against
exactly the same held-out set never redrawn.

Degenerate (zero-range) dimensions e.g. Push/Slide's table-height `z`, which is fixed,
not sampled are left untouched by the direction/margin logic below (a `free`-dims mask),
so a zone center never drifts off an axis the task doesn't actually vary.

If a curriculum proposes a training target that happens to fall inside a reserved zone,
`enforce_held_out_exclusion()` does **not** clip it to the nearest valid point (that would
bias the training distribution towards the zone boundary); instead it locally jitters and
rechecks, falling back to a fresh uniform resample if needed.

In [3]:
HELD_OUT_SEED = 20260921  # fixed once; do not change between runs/conditions
N_ZONES_PER_TASK = 5
ZONE_RADIUS_FRACTION = 0.08   # zone radius, as a fraction of the task's own range half-diagonal
ZONE_DIFFICULTY_LEVELS = ["near", "near", "medium", "medium", "far"]
DIFFICULTY_TO_FRACTION = {"near": 0.20, "medium": 0.55, "far": 0.85}  # fraction of half-diagonal from centre


def _sample_direction(rng: np.random.RandomState, dim: int) -> np.ndarray:
    v = rng.normal(size=dim)
    n = np.linalg.norm(v)
    return v / n if n > 1e-9 else v


def build_held_out_zones() -> Dict[str, List[Dict]]:
    """Returns {task_name: [{'center': np.ndarray, 'radius': float,radius_eval':float 'level': str}, ...]}."""
    rng = np.random.RandomState(HELD_OUT_SEED)
    zones: Dict[str, List[Dict]] = {}
    for name in TASK_NAMES:
        spec = TASK_SPECS[name]
        low, high = spec.goal_low, spec.goal_high
        free = (high - low) > 1e-9          # which dims actually vary for this task
        center = (low + high) / 2.0
        half_extent = (high - low) / 2.0
        half_diag = np.linalg.norm(half_extent[free]) if free.any() else 0.0
        radius = ZONE_RADIUS_FRACTION * half_diag if half_diag > 0 else 1e-3
        radius_eval=radius/2
        task_zones = []
        for level in ZONE_DIFFICULTY_LEVELS:
            frac = DIFFICULTY_TO_FRACTION[level]
            direction = np.zeros(spec.goal_dim)
            direction[free] = _sample_direction(rng, int(free.sum()))
            point = center + direction * frac * half_diag
            margin = np.where(free, radius * 0.5, 0.0)
            point = np.clip(point, low + margin, high - margin)
            task_zones.append({"center": point.astype(np.float32), "radius": float(radius),"radius_eval":float(radius_eval), "level": level})
        zones[name] = task_zones
    return zones


HELD_OUT_ZONES = build_held_out_zones()


def in_held_out_zone(name: str, point: np.ndarray) -> bool:
    for zone in HELD_OUT_ZONES[name]:
        if np.linalg.norm(point - zone["center"]) < zone["radius"]:
            return True
    return False


def enforce_held_out_exclusion(name: str, point: np.ndarray, rng: np.random.Generator,
                                max_tries: int = 50) -> np.ndarray:
    """Section 6: local jitter + recheck (never clip-to-nearest), falling back to a fresh
    uniform resample if the jitter loop doesn't escape within max_tries."""
    spec = TASK_SPECS[name]
    low, high = spec.goal_low, spec.goal_high
    p = point.copy()
    jitter_scale = 2.0 * max(z["radius"] for z in HELD_OUT_ZONES[name])
    for _ in range(max_tries):
        if not in_held_out_zone(name, p):
            return p
        p = point + rng.normal(scale=jitter_scale, size=point.shape)
        p = np.clip(p, low, high)
    for _ in range(max_tries):
        p = rng.uniform(low, high)
        if not in_held_out_zone(name, p):
            return p
    raise RuntimeError(f"Could not find a valid point outside held-out zones for task {name}")


for name in TASK_NAMES:
    levels = [z["level"] for z in HELD_OUT_ZONES[name]]
    print(f"{name:14s} {N_ZONES_PER_TASK} zones, levels={levels}, "
          f"radius={HELD_OUT_ZONES[name][0]['radius']:.4f}")


reach          5 zones, levels=['near', 'near', 'medium', 'medium', 'far'], radius=0.0208
push           5 zones, levels=['near', 'near', 'medium', 'medium', 'far'], radius=0.0170
slide          5 zones, levels=['near', 'near', 'medium', 'medium', 'far'], radius=0.0170
pickandplace   5 zones, levels=['near', 'near', 'medium', 'medium', 'far'], radius=0.0188
stack          5 zones, levels=['near', 'near', 'medium', 'medium', 'far'], radius=0.0170


In [4]:
SHARED_ACTION_DIM = 4  # 3D end-effector displacement + 1D gripper (sliced off for block_gripper tasks)

def pad_goal(g: np.ndarray, name: str) -> np.ndarray:
    g = np.asarray(g, dtype=np.float32).reshape(-1)

    if g.shape[0]== MAX_GOAL_DIM:
        return g
    new_g=np.zeros( MAX_GOAL_DIM,dtype=np.float32)
    new_g[:g.shape[0]]=g
    return new_g
    
def pad_robot_obs(x: np.ndarray) -> np.ndarray:

    x = np.asarray(x, dtype=np.float32).reshape(-1)

    if x.shape[0]== MAX_ROBOT_OBS_DIM:
        return x
    new_x=np.zeros(MAX_ROBOT_OBS_DIM,dtype=np.float32)
    new_x[:x.shape[0]]=x
    return new_x

def one_hot_task(task_id: int) -> np.ndarray:
    tab=np.zeros(N_TASKS,dtype=np.float32)
    tab[task_id]=1.0
    return tab
    
def shared_compute_reward(achieved_goal, desired_goal, info) -> np.ndarray:

    achieved_goal= np.asarray(achieved_goal,dtype=np.float32)
    desired_goal= np.asarray(desired_goal,dtype=np.float32)

    #if not batch
    single=desired_goal.ndim==1

    if single:
        achieved_goal=achieved_goal[None,:]
        desired_goal=desired_goal[None,:]
        info=[info]
        
    n=desired_goal.shape[0]
    rewards = np.zeros(n, dtype=np.float32)
    for i in range(n):
        info_i=info[i]
        task_id = int(info_i["task_id"]) if isinstance(info_i, dict) else int(info_i)
        name=TASK_NAMES[task_id]
        gd = native_goal_dim(name)
        thr = TASK_SPECS[name].threshold
        d = np.linalg.norm(achieved_goal[i, :gd] - desired_goal[i, :gd])
        rewards[i] = -float(d > thr)
    return rewards[0] if single else rewards
        
        
    
    
def shared_is_success(achieved_goal_padded, desired_goal_padded, task_id: int) -> bool:
    name = TASK_NAMES[task_id]
    gd = native_goal_dim(name)
    thr = TASK_SPECS[name].threshold
    d = np.linalg.norm(np.asarray(achieved_goal_padded)[:gd] - np.asarray(desired_goal_padded)[:gd])
    return bool(d < thr)



## Non-LLM Baselines 

1) Random
2) Hand-Crafted 1
3) Hand-Crafted 2
4) Adaptive (UCB)

In [5]:
class Curriculum:
    def propose(self, rng: np.random.Generator) -> Tuple[int, np.ndarray]:
        raise NotImplementedError

    def notify_result(self, task_id: int, success: bool) -> None:
        """Optional hook, called after each episode ends (task_id, whether it succeeded)."""
        pass


class RandomCurriculum(Curriculum):
    """Baseline 1: task drawn uniformly at random; target drawn uniformly from the task's
    full native curriculum-space range."""

    def propose(self, rng: np.random.Generator) -> Tuple[int, np.ndarray]:
        selected_task = int(rng.integers(0, N_TASKS))
        name = TASK_NAMES[selected_task]
        spec = TASK_SPECS[name]
        target = rng.uniform(spec.goal_low, spec.goal_high)
        return spec.task_id, target.astype(np.float32)


class HandCraftedCurriculum(Curriculum):
    """Baseline 2 (naive): task order easy-to-hard (reach -> push -> slide -> pickandplace ->
    stack), advancing on a rolling success rate > ADVANCE_THRESHOLD over WINDOW episodes.
    Within a task, the sampled target distance from the range's centre shrinks linearly from
    far (task start) to near (task end)."""

    ADVANCE_THRESHOLD = 0.70
    WINDOW = 20

    def __init__(self):
        self.order = ["reach", "push", "slide", "pickandplace", "stack"]
        self.stage = 0
        self.history: List[bool] = []

    def _rolling_success(self) -> float:
        if not self.history:
            return 0.0
        recent = self.history[-self.WINDOW:]
        return float(np.mean(recent))

    def propose(self, rng: np.random.Generator) -> Tuple[int, np.ndarray]:
        task = self.order[self.stage]
        spec = TASK_SPECS[task]
        low, high = spec.goal_low, spec.goal_high
        center = (low + high) / 2.0

        progress = np.clip(self._rolling_success() / self.ADVANCE_THRESHOLD, 0.0, 1.0)
        frac = max(1.0 - progress, 0.15)
        goal_low = center - frac * (center - low)
        goal_high = center + frac * (high - center)
        target = rng.uniform(goal_low, goal_high)

        return spec.task_id, target.astype(np.float32)

    def notify_result(self, task_id: int, success: bool) -> None:
        name = TASK_NAMES[task_id]
        if name != self.order[self.stage]:
            return  # stale feedback from before the last advance; ignore
        self.history.append(success)
        if self._rolling_success() >= self.ADVANCE_THRESHOLD and len(self.history) >= self.WINDOW:
            if self.stage < len(self.order) - 1:
                self.stage += 1
                self.history = []


class HandCraftedCurriculum2(Curriculum):
    """Baseline 3 (domain-informed): groups Reach, Push and Slide together as the
    non-grasping skill family and interleaves training among them from the start,
    introduces Pick-and-place once that family is mastered, and starts allocating a
    growing share of episodes to Stack.
    """

    ADVANCE_THRESHOLD = 0.70        # success rate needed for reach/push/slide to be "mastered"
    THRESHOLD_START_STACK = 0.50    # lower success rate on pickandplace, enough to START introducing stack
    WINDOW = 20
    NON_GRASPING = ["reach", "push", "slide"]
    STACK_RAMP_EPISODES = 100       # episodes over which Stack's share grows from 0% to STACK_MAX_SHARE
    STACK_MAX_SHARE = 0.70          # Stack never fully replaces Pick-and-place practice

    def __init__(self):
        self.order = ["reach", "push", "slide", "pickandplace", "stack"]
        self.history: Dict[str, List[bool]] = {name: [] for name in TASK_NAMES}
        self.family_done = False        # reach/push/slide all mastered (>= ADVANCE_THRESHOLD)
        self.stack_unlocked = False     # pickandplace reached THRESHOLD_START_STACK -> ok to start stack
        self.stack_episode_count = 0    # episodes elapsed since stack_unlocked became True

    def _rolling_success(self, name: str) -> float:
        h = self.history[name]
        if not h:
            return 0.0
        recent = h[-self.WINDOW:]
        return float(np.mean(recent))

    def _is_mastered(self, name: str, threshold: float) -> bool:
       
        return self._rolling_success(name) >= threshold and len(self.history[name]) >= self.WINDOW

    def propose(self, rng: np.random.Generator) -> Tuple[int, np.ndarray]:
        if not self.family_done:
            name = self.NON_GRASPING[int(rng.integers(0, len(self.NON_GRASPING)))]
        elif not self.stack_unlocked:
            name = "pickandplace"
        else:
            stack_share = min(self.stack_episode_count / self.STACK_RAMP_EPISODES, self.STACK_MAX_SHARE)
            name = "stack" if rng.random() < stack_share else "pickandplace"

        spec = TASK_SPECS[name]
        low, high = spec.goal_low, spec.goal_high
        center = (low + high) / 2.0
        progress = np.clip(self._rolling_success(name) / self.ADVANCE_THRESHOLD, 0.0, 1.0)
        frac = max(1.0 - progress, 0.15)
        goal_low = center - frac * (center - low)
        goal_high = center + frac * (high - center)
        target = rng.uniform(goal_low, goal_high)
        return spec.task_id, target.astype(np.float32)

    def notify_result(self, task_id: int, success: bool) -> None:
        name = TASK_NAMES[task_id]
        self.history[name].append(success)

        if not self.family_done and all(self._is_mastered(n, self.ADVANCE_THRESHOLD) for n in self.NON_GRASPING):
            self.family_done = True

        if self.family_done and not self.stack_unlocked and self._is_mastered("pickandplace", self.THRESHOLD_START_STACK):
            self.stack_unlocked = True

        if self.stack_unlocked:
            self.stack_episode_count += 1


class UCBAdaptiveCurriculum(Curriculum):
    """Baseline 5 (adaptive, non-LLM): discretized UCB1 bandit over (task, distance-level)
    bins, weighted by absolute learning progress (Graves et al., 2017)."""

    DIFFICULTY_LEVELS = ["near", "medium", "far"]
    UCB_C = 2.0          # exploration/exploitation trade-off constant
    WINDOW = 20          # how many recent episodes define "recent" success rate per bin
    N_BINS = N_TASKS * len(DIFFICULTY_LEVELS)  # 5 tasks x 3 levels = 15 bins
    DIFFICULTY_TO_FRACTION=[0.20,0.55,0.80]

    def __init__(self):
        # a "bin" (task_name, difficulty_level) 
        self.bins = [(name, level) for name in TASK_NAMES for level in self.DIFFICULTY_LEVELS]

        self.n_pulls = np.zeros(self.N_BINS, dtype=np.int64)   # n_i : how many times this bin was chosen
        self.history = {i: [] for i in range(self.N_BINS)}     # success history, per bin
        self.total_pulls = 0

    def _learning_progress(self, bin_idx: int) -> float:

        if len(self.history[bin_idx])<20:
            return 0.0
        
        else:
            old=np.mean(self.history[bin_idx][-20:-10])
            new=np.mean(self.history[bin_idx][-10:])
        return np.abs(old-new)
            
            

    def _ucb_score(self, bin_idx: int) -> float:

        if self.n_pulls[bin_idx]==0:
            return float("inf")
        return  self._learning_progress(bin_idx)+ self.UCB_C*np.sqrt(np.log(self.total_pulls) / self.n_pulls[bin_idx])
            
        

    def propose(self, rng: np.random.Generator) -> Tuple[int, np.ndarray]:
      
        bin_idx = np.argmax([self._ucb_score(i) for i in range(self.N_BINS)])
        name, level = self.bins[bin_idx]
        spec = TASK_SPECS[name]
        center = (spec.goal_low + spec.goal_high) / 2.0
    
            
        frac_center = self.DIFFICULTY_TO_FRACTION[self.DIFFICULTY_LEVELS.index(level)]        
        band = 0.15
        frac = rng.uniform(max(frac_center - band, 0.0), min(frac_center + band, 1.0))
    
        low = center - frac * (center - spec.goal_low)
        high = center + frac * (spec.goal_high - center)
        target = rng.uniform(low, high)
    
        self._last_bin = bin_idx
        return spec.task_id, target.astype(np.float32)
            

    def notify_result(self, task_id: int, success: bool) -> None:
    
        self.n_pulls[self._last_bin]+=1
        self.total_pulls+=1
        self.history[self._last_bin].append(success)

### MultiTaskCurriculumEnv

Unifies the 5 native panda-gym environments into a single `gym.Env` driven by a
`Curriculum` and the held-out exclusion mechanism: every `reset()` asks the curriculum
for `(task_id, target)`, checks it isn't inside a reserved held-out zone (jittering it
away if it is), overrides the underlying task's goal directly, and returns the shared,
padded observation. `set_forced_episode()` lets the held-out evaluator bypass the
curriculum entirely and force a specific, exact held-out zone.

In [6]:
class MultiTaskCurriculumEnv(gym.Env):
    metadata = {"render_modes": []}

    def __init__(self, curriculum: Curriculum, seed: int = 0, held_out_eval: bool = False):
        super().__init__()
        self.curriculum = curriculum
        self.held_out_eval = held_out_eval  
        self._forced_episode: Optional[Tuple[int, np.ndarray]] = None
        self.rng = np.random.default_rng(seed)

        self.envs = {name: gym.make(TASK_ENV_IDS[name]) for name in TASK_NAMES}
        for e in self.envs.values():
            e.reset(seed=int(self.rng.integers(0, 2**31 - 1)))

        self.action_space = spaces.Box(-1.0, 1.0, shape=(SHARED_ACTION_DIM,), dtype=np.float32)
        obs_dim = MAX_ROBOT_OBS_DIM + N_TASKS
        self.observation_space = spaces.Dict(
            dict(
                observation=spaces.Box(-10.0, 10.0, shape=(obs_dim,), dtype=np.float32),
                achieved_goal=spaces.Box(-10.0, 10.0, shape=(MAX_GOAL_DIM,), dtype=np.float32),
                desired_goal=spaces.Box(-10.0, 10.0, shape=(MAX_GOAL_DIM,), dtype=np.float32),
            )
        )
        self.compute_reward = shared_compute_reward 

        self.active_task_id: Optional[int] = None
        self._elapsed = 0

    def _set_goal(self, name: str, curriculum_target: np.ndarray) -> None:
        env = self.envs[name].unwrapped
        spec = TASK_SPECS[name]
        if name == "stack":
            goal6 = expand_stack_goal(curriculum_target, spec)
            env.task.goal = goal6
            env.sim.set_base_pose("target1", goal6[:3], np.array([0.0, 0.0, 0.0, 1.0]))
            env.sim.set_base_pose("target2", goal6[3:], np.array([0.0, 0.0, 0.0, 1.0]))
        else:
            env.task.goal = curriculum_target.astype(np.float32)
            env.sim.set_base_pose("target", curriculum_target, np.array([0.0, 0.0, 0.0, 1.0]))

    def _build_obs(self, name: str) -> Dict[str, np.ndarray]:
        task_id = TASK_SPECS[name].task_id
        env = self.envs[name].unwrapped
        x = pad_robot_obs(env.robot.get_obs()[: (7 if not TASK_SPECS[name].block_gripper else 6)])
        achieved = pad_goal(env.task.get_achieved_goal(), name)
        desired = pad_goal(env.task.get_goal(), name)
        return {
            "observation": np.concatenate([x, one_hot_task(task_id)]).astype(np.float32),
            "achieved_goal": achieved,
            "desired_goal": desired,
        }

    def set_forced_episode(self, task_id: int, target: np.ndarray) -> None:
        """Used by the held-out evaluator to force a specific (task, target) at the next reset(),
        bypassing the curriculum and the held-out exclusion (evaluation is ON the held-out set)."""
        self._forced_episode = (task_id, target)

    def reset(self, *, seed: Optional[int] = None, options: Optional[dict] = None):
        if self._forced_episode is not None:
            task_id, target = self._forced_episode
            self._forced_episode = None
        else:
            task_id, target = self.curriculum.propose(self.rng)
            name = TASK_NAMES[task_id]
            target = enforce_held_out_exclusion(name, target, self.rng)

        name = TASK_NAMES[task_id]
        self.active_task_id = task_id
        self.envs[name].reset()
        self._set_goal(name, target)
        obs = self._build_obs(name)
        self._elapsed = 0
        success = shared_is_success(obs["achieved_goal"], obs["desired_goal"], task_id)
        info = {"is_success": success, "task_id": task_id}
        return obs, info

    def step(self, action: np.ndarray):
        name = TASK_NAMES[self.active_task_id]
        spec = TASK_SPECS[name]
        native_action_dim = 3 if spec.block_gripper else 4
        sliced = np.asarray(action, dtype=np.float32)[:native_action_dim]
        _native_obs, _native_reward, terminated, truncated, _native_info = self.envs[name].step(sliced)

        obs = self._build_obs(name)
        reward = float(shared_compute_reward(obs["achieved_goal"], obs["desired_goal"], {"task_id": self.active_task_id}))
        success = reward > -0.5
        terminated = bool(success)
        info = {"is_success": success, "task_id": self.active_task_id}
        self._elapsed += 1

        self.curriculum.notify_result(self.active_task_id, success) if (terminated or truncated) else None
        return obs, reward, terminated, truncated, info

    def close(self):
        for e in self.envs.values():
            e.close()

print("MultiTaskCurriculumEnv defined.")


MultiTaskCurriculumEnv defined.


### Training and Evaluation 

In [7]:
N_EVAL_EPISODES_PER_ZONE = 5
EVAL_FREQ=1000
TOTAL_TIMESTEPS= 250_000
HER_KWARGS = dict(n_sampled_goal=4, goal_selection_strategy="future", copy_info_dict=True)
SAC_KWARGS = dict(
    learning_rate=1e-3,
    buffer_size=200_000,
    learning_starts=1_000,
    batch_size=256,
    train_freq=1,
    gradient_steps=1,
    gamma=0.95,
    tau=0.05,
    policy_kwargs=dict(net_arch=[256, 256]),
)


def evaluate_held_out(model, eval_env: "MultiTaskCurriculumEnv", n_episodes_per_zone: int = N_EVAL_EPISODES_PER_ZONE,
                       deterministic: bool = True) -> Dict[str, float]:
    per_task: Dict[str, float] = {}
    for name in TASK_NAMES:
        spec = TASK_SPECS[name]
        free = (spec.goal_high - spec.goal_low) > 1e-9
        successes = []
        for zone in HELD_OUT_ZONES[name]:
            for _ in range(n_episodes_per_zone):
               
                direction = np.zeros(spec.goal_dim)
                direction[free] = _sample_direction(eval_env.rng, int(free.sum()))
                magnitude = eval_env.rng.uniform(0.0, zone["radius_eval"])
                target = (zone["center"] + direction * magnitude).astype(np.float32)

                eval_env.set_forced_episode(spec.task_id, target)
                obs, info = eval_env.reset()
                done = False
                while not done:
                    action, _ = model.predict(obs, deterministic=deterministic)
                    obs, reward, terminated, truncated, info = eval_env.step(action)
                    done = terminated or truncated
                successes.append(bool(info["is_success"]))
        per_task[name] = float(np.mean(successes))
    return per_task


from stable_baselines3.common.callbacks import BaseCallback


class HeldOutEvalCallback(BaseCallback):
    """Periodic held-out evaluation, logging one success-rate-per-task point every eval_freq steps."""

    def __init__(self, eval_env: "MultiTaskCurriculumEnv", eval_freq: int,
                 n_episodes_per_zone: int = N_EVAL_EPISODES_PER_ZONE, verbose: int = 0):
        super().__init__(verbose)
        self.eval_env = eval_env
        self.eval_freq = eval_freq
        self.n_episodes_per_zone = n_episodes_per_zone
        self.history: List[Dict] = []

    def _on_step(self) -> bool:
        if self.n_calls % self.eval_freq == 0:
            per_task = evaluate_held_out(self.model, self.eval_env, self.n_episodes_per_zone)
            entry = {"timesteps": self.num_timesteps, **{f"success_{k}": v for k, v in per_task.items()}}
            self.history.append(entry)
            if self.verbose:
                print(f"  [held-out eval @ {self.num_timesteps:>7d}] " +
                      " ".join(f"{k}={v:.2f}" for k, v in per_task.items()))
        return True


def make_model(env: "MultiTaskCurriculumEnv", seed: int):
    from stable_baselines3 import SAC
    from stable_baselines3.her import HerReplayBuffer
    return SAC(
        "MultiInputPolicy",
        env,
        replay_buffer_class=HerReplayBuffer,
        replay_buffer_kwargs=HER_KWARGS,
        seed=seed,
        verbose=0,
        **SAC_KWARGS,
    )


def train_one_run(curriculum_factory, seed: int, total_timesteps: int, eval_freq: int,
                   n_episodes_per_zone: int = N_EVAL_EPISODES_PER_ZONE, verbose: int = 0) -> List[Dict]:
    """curriculum_factory: zero-arg callable returning a fresh Curriculum instance
    (e.g. RandomCurriculum, HandCraftedCurriculum)."""
    from stable_baselines3.common.utils import set_random_seed
    set_random_seed(seed)

    train_env = MultiTaskCurriculumEnv(curriculum_factory(), seed=seed)
    eval_env = MultiTaskCurriculumEnv(RandomCurriculum(), seed=seed + 10_000, held_out_eval=True)

    model = make_model(train_env, seed=seed)
    cb = HeldOutEvalCallback(eval_env, eval_freq=eval_freq, n_episodes_per_zone=n_episodes_per_zone, verbose=verbose)
    model.learn(total_timesteps=total_timesteps, callback=cb, progress_bar=False)

    train_env.close()
    eval_env.close()
    return cb.history

print("Held-out evaluation + SAC+HER training harness defined.")


Held-out evaluation + SAC+HER training harness defined.


In [ ]:
import json
import os
import time

RESULTS_PATH = "results_phase1.json"
Curriculums = {
    "random": RandomCurriculum,
    "hand crafted 1(naive)": HandCraftedCurriculum,
    "hand crafted 2 (domain-informed)": HandCraftedCurriculum2,
    "Adaptive": UCBAdaptiveCurriculum,
}
STRATEGIES = list(Curriculums.keys())  


def _overall_success(entry: Dict) -> float:
    """Mean success rate across the 5 tasks for one held-out-eval checkpoint entry."""
    return float(np.mean([v for k, v in entry.items() if k.startswith("success_")]))


if os.path.exists(RESULTS_PATH):
    with open(RESULTS_PATH) as f:
        results = json.load(f)
    for strategy in STRATEGIES:
        results.setdefault(strategy, {})
else:
    results = {strategy: {} for strategy in STRATEGIES}

t0 = time.time()
for name, c in Curriculums.items():
    for seed in SEEDS:
        seed_key = str(seed)
        if seed_key in results[name]:  
            print(f"skip strategy={name!r} seed={seed} (already done)")
            print(f"success rate (mean over tasks): {_overall_success(results[name][seed_key][-1]):.2f}")
            continue
        print(f"strategy={name}  seed={seed}  ({time.time() - t0:.0f}s elapsed)")
        history = train_one_run(c, seed, total_timesteps=TOTAL_TIMESTEPS, eval_freq=EVAL_FREQ,
                                 n_episodes_per_zone=N_EVAL_EPISODES_PER_ZONE, verbose=0)
        results[name][seed_key] = history
        json.dump(results, open(RESULTS_PATH, "w"), indent=2)
        print(f"success rate (mean over tasks): {_overall_success(history[-1]):.2f}")

print(f"End in {time.time() - t0:.0f}s")

skip strategy='random' seed=10 (already done)
success rate (mean over tasks): 0.30
strategy=random  seed=11  (0s elapsed)


pybullet build time: Jan 29 2025 23:17:20


success rate (mean over tasks): 0.26
strategy=random  seed=12  (8814s elapsed)
success rate (mean over tasks): 0.24
strategy=random  seed=13  (17735s elapsed)
argv[0]=--background_color_red=0.8745098114013672
argv[1]=--background_color_green=0.21176470816135406
argv[2]=--background_color_blue=0.1764705926179886
argv[0]=--background_color_red=0.8745098114013672
argv[1]=--background_color_green=0.21176470816135406
argv[2]=--background_color_blue=0.1764705926179886
argv[0]=--background_color_red=0.8745098114013672
argv[1]=--background_color_green=0.21176470816135406
argv[2]=--background_color_blue=0.1764705926179886
argv[0]=--background_color_red=0.8745098114013672
argv[1]=--background_color_green=0.21176470816135406
argv[2]=--background_color_blue=0.1764705926179886
argv[0]=--background_color_red=0.8745098114013672
argv[1]=--background_color_green=0.21176470816135406
argv[2]=--background_color_blue=0.1764705926179886
argv[0]=--background_color_red=0.8745098114013672
argv[1]=--background_

In [ ]:
# Build a tidy (long-format) dataframe from the in-memory `results` dict:
# one row per (strategy, seed, checkpoint, task).
rows = []
for strategy, seeds in results.items():
    for seed_key, history in seeds.items():
        for entry in history:
            for task in TASK_NAMES:
                rows.append({
                    "strategy": strategy,
                    "seed": int(seed_key),
                    "timesteps": entry["timesteps"],
                    "task": task,
                    "success": entry[f"success_{task}"],
                })
df = pd.DataFrame(rows)


df_global = (
    df.groupby(["strategy", "seed", "timesteps"], as_index=False)["success"]
      .mean()
)

print(df.shape, df_global.shape)
df.head()

In [ ]:
COLORS = {
    "random": "#2a78d6",
    "hand crafted 1(naive)": "#eb6834",
    "hand crafted 2 (domain-informed)": "#1baf7a",
    "Adaptive": "#eda100",
}
LABELS = {
    "random": "Random goals",
    "hand crafted 1(naive)": "Hand-crafted 1 (naive)",
    "hand crafted 2 (domain-informed)": "Hand-crafted 2 (domain-informed)",
    "Adaptive": "Adaptive (UCB)",
}

fig, ax = plt.subplots(figsize=(8, 5))

for strategy in STRATEGIES:
    sub = df_global[df_global["strategy"] == strategy]
    if sub.empty:
        continue
    agg = sub.groupby("timesteps").agg(mean=("success", "mean"), std=("success", "std")).reset_index()

    color, label = COLORS[strategy], LABELS[strategy]
    ax.plot(agg["timesteps"], agg["mean"], label=label, color=color, linewidth=2)
    if agg["std"].notna().any() and sub["seed"].nunique() > 1:
        ax.fill_between(agg["timesteps"],
                         (agg["mean"] - agg["std"]).clip(lower=0),
                         (agg["mean"] + agg["std"]).clip(upper=1),
                         color=color, alpha=0.15)

ax.set_xlabel("Training steps")
ax.set_ylabel("Success rate (mean over the 5 tasks)")
ax.set_ylim(-0.02, 1.02)
ax.set_title("Overall held-out success rate vs. training time, by curriculum")
ax.grid(alpha=0.3)
ax.legend(fontsize=9)
fig.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharex=True, sharey=True)
axes = axes.ravel()

for i, task in enumerate(TASK_NAMES):
    ax = axes[i]
    sub_task = df[df["task"] == task]
    for strategy in STRATEGIES:
        sub = sub_task[sub_task["strategy"] == strategy]
        if sub.empty:
            continue
        agg = sub.groupby("timesteps").agg(mean=("success", "mean"), std=("success", "std")).reset_index()
        color, label = COLORS[strategy], LABELS[strategy]
        ax.plot(agg["timesteps"], agg["mean"], label=label, color=color, linewidth=2)
        if agg["std"].notna().any() and sub["seed"].nunique() > 1:
            ax.fill_between(agg["timesteps"],
                             (agg["mean"] - agg["std"]).clip(lower=0),
                             (agg["mean"] + agg["std"]).clip(upper=1),
                             color=color, alpha=0.15)
    ax.set_title(task)
    ax.set_ylim(-0.02, 1.02)
    ax.grid(alpha=0.3)

axes[-1].axis("off")  # only 5 tasks for a 6-slot grid
axes[0].set_ylabel("Success rate")
axes[3].set_ylabel("Success rate")
for ax in axes[3:5]:
    ax.set_xlabel("Training steps")

handles, labels_ = axes[0].get_legend_handles_labels()
fig.legend(handles, labels_, loc="lower center", ncol=4, fontsize=9, bbox_to_anchor=(0.5, -0.02))
fig.suptitle("Held-out success rate per task vs. training time, by curriculum", fontsize=13)
fig.tight_layout(rect=[0, 0.04, 1, 1])
plt.show()